In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# -----------------------------
# 1. Cargar y normalizar datos
# -----------------------------
transform = transforms.ToTensor()   # Ya normaliza a [0, 1]

train_dataset = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

print(f"Imágenes de entrenamiento: {len(train_dataset)}")
print(f"Imágenes de test: {len(test_dataset)}")

# Comprobamos la normalización
img, label = train_dataset[0]
print(f"Forma de la imagen: {img.shape}")
print(f"Valor mínimo: {img.min().item():.3f}")
print(f"Valor máximo: {img.max().item():.3f}")

Imágenes de entrenamiento: 60000
Imágenes de test: 10000
Forma de la imagen: torch.Size([1, 28, 28])
Valor mínimo: 0.000
Valor máximo: 1.000


In [5]:
# -----------------------------
# 2. Convertir etiquetas a one-hot
# -----------------------------
def to_one_hot(labels, num_classes=10):
    return F.one_hot(labels, num_classes=num_classes).float()

# Ejemplo con una sola etiqueta
etiqueta_ejemplo = torch.tensor(7)          # 7 = Sneaker
one_hot_ejemplo = to_one_hot(etiqueta_ejemplo)

print("Etiqueta original:", etiqueta_ejemplo.item())
print("One-hot:", one_hot_ejemplo)

Etiqueta original: 7
One-hot: tensor([0., 0., 0., 0., 0., 0., 0., 1., 0., 0.])


In [6]:
# -----------------------------
# 3. Estructura básica de la red
# -----------------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.flatten = nn.Flatten()          # [1, 28, 28] → [784]
        
        self.network = nn.Sequential(
            nn.Linear(784, 128),             # Capa oculta 1
            nn.ReLU(),
            nn.Linear(128, 64),              # Capa oculta 2
            nn.ReLU(),
            nn.Linear(64, 10)                # Capa de salida (10 clases)
        )
    
    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)

# Creamos el modelo
model = MLP()
print(model)

# Contamos parámetros
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal de parámetros entrenables: {total_params:,}")

MLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

Total de parámetros entrenables: 109,386


In [7]:
# Probamos que la red acepta una imagen
img, label = train_dataset[0]
img = img.unsqueeze(0)          # Añadimos dimensión de batch → [1, 1, 28, 28]

with torch.no_grad():
    salida = model(img)

print("Forma de la salida de la red:", salida.shape)   # Debe ser [1, 10]
print("Valores de salida (logits):", salida)

Forma de la salida de la red: torch.Size([1, 10])
Valores de salida (logits): tensor([[ 0.0102, -0.0126,  0.0113, -0.1173, -0.0434,  0.1025,  0.1856, -0.0985,
         -0.0021, -0.1128]])
